# Transfert et fine-tuning mesurables

Préentraînement local sur digits 0–4, transfert vers 5–9. CPU, sans téléchargement. Voir ateliers_avances.md pour le protocole et les limites. Les budgets sont fixés avant le test ; aucun vainqueur n’est imposé.

## 1. Données et modèle

Les classes source et cible sont disjointes. Le train cible est limité à 150 images ; le reste forme un test indépendant. Un MLP sert d’extracteur pour isoler le mécanisme de transfert.

In [1]:
import copy
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
torch.set_num_threads(1); torch.manual_seed(42)
data = load_digits()
x = torch.tensor(data.data/16,dtype=torch.float32)
y = torch.tensor(data.target,dtype=torch.long)
source_x,source_y = x[y<5],y[y<5]
target_x,target_y = x[y>=5],y[y>=5]-5
train_ids,test_ids = train_test_split(np.arange(len(target_y)),train_size=150,stratify=target_y.numpy(),random_state=42)
assert set(train_ids).isdisjoint(test_ids)
def network():
    return nn.Sequential(nn.Linear(64,32),nn.ReLU(),nn.Linear(32,5))
def train(model,inputs,targets,optimizer,steps):
    model.train()
    for _ in range(steps):
        loss = nn.functional.cross_entropy(model(inputs),targets)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    return loss.item()
source = network()
train(source,source_x,source_y,torch.optim.Adam(source.parameters(),lr=.01),180)

0.004506378900259733

## 2. Trois adaptations et contrôle des poids

La tête cible est réinitialisée identiquement pour les variantes. Le fine-tuning utilise un plus petit taux pour l’extracteur. On mesure les poids réellement modifiés.

In [2]:
results = []
for mode in ['zéro','figé','fine-tuning']:
    torch.manual_seed(15)
    model = network()
    if mode != 'zéro':
        model[0].load_state_dict(copy.deepcopy(source[0].state_dict()))
    before = model[0].weight.detach().clone()
    if mode == 'figé':
        for p in model[0].parameters(): p.requires_grad_(False)
    optimizer = torch.optim.Adam([
        {'params':model[0].parameters(),'lr':.002 if mode=='fine-tuning' else .01},
        {'params':model[2].parameters(),'lr':.01}])
    final_loss = train(model,target_x[train_ids],target_y[train_ids],optimizer,180)
    changed = not torch.equal(before,model[0].weight)
    assert changed == (mode != 'figé')
    model.eval()
    with torch.no_grad():
        accuracy = (model(target_x[test_ids]).argmax(1)==target_y[test_ids]).float().mean().item()
    results.append({'stratégie':mode,'exactitude test':accuracy,'loss train':final_loss,'extracteur modifié':changed})
print(pd.DataFrame(results).set_index('stratégie'))

             exactitude test  loss train  extracteur modifié
stratégie                                                   
zéro                0.966488    0.001747                True
figé                0.855228    0.423318               False
fine-tuning         0.958445    0.056277                True


## Exercice

Réduisez le nombre de labels cibles puis répétez avec plusieurs graines. Est-ce que figer signifie simplement ne pas appeler backward ?

In [3]:
# Écrivez votre expérience ici avant de lire la correction.

## Correction et limites

Non : backward est nécessaire pour entraîner la tête ; requires_grad=False bloque les gradients des paramètres figés. Avec BatchNorm, le mode train peut modifier des statistiques même sans gradients ; ce MLP n’en contient pas. Mesurer une distribution de scores sans sélectionner la meilleure graine sur test.